# Contraste Scope / CSV y escenarios · Polipasto manual
Ejecuta las celdas en orden (Shift+Enter). Requiere `pandas` y `matplotlib`.

1. Compara `datos_mecanismo.csv` (Simulink) con la solución analítica.
2. Superpone los escenarios de la carpeta `escenarios/` (como en el Taller 2).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

datos = pd.read_csv('datos_mecanismo.csv')
print(datos.shape)
datos.head()

## 1. Revisión del CSV
La primera columna debe ser `t [s]` y deben salir 2001 filas (0 a 20 s cada 0.01 s).

In [ ]:
assert datos.columns[0] == 't [s]', 'La primera columna debe ser t [s]'
print('Columnas:', list(datos.columns))
print('dt =', np.round(np.diff(datos['t [s]']).mean(), 6), 's')

## 2. Simulink frente a la solución analítica
`referencia_analitica_nominal.csv` sale de las fórmulas cerradas de la Fase I (caso nominal). Los errores deben ser prácticamente cero (del orden de 1e-6 o menos).

In [ ]:
ref = pd.read_csv('referencia_analitica_nominal.csv')
for col in ['x [m]', 'v [m/s]', 'a [m/s^2]', 'T [N]', 'P [W]', 'W_op [J]']:
    e = datos[col].values - ref[col].values
    print(f'{col:12s}  error max = {np.abs(e).max():.2e}   RMS = {np.sqrt((e**2).mean()):.2e}')

## 3. Las mismas gráficas del Scope, dibujadas desde el CSV
Guarda esta figura junto a la captura del Scope para la memoria.

In [ ]:
t = datos['t [s]']
fig, ax = plt.subplots(6, 1, figsize=(9, 12), sharex=True)
ax[0].plot(t, datos['x [m]']); ax[0].set_ylabel('x [m]')
ax[1].plot(t, datos['v [m/s]']); ax[1].set_ylabel('v [m/s]')
ax[2].plot(t, datos['a [m/s^2]']); ax[2].set_ylabel('a [m/s²]')
ax[3].plot(t, datos['T [N]'], label='T = F_op'); ax[3].axhline(300, ls='--', c='r', label='F_max')
ax[3].set_ylabel('[N]'); ax[3].legend()
ax[4].plot(t, datos['P [W]']); ax[4].set_ylabel('P [W]')
ax[5].plot(t, datos['W_op [J]'], label='W_op'); ax[5].plot(t, datos['E_mec [J]'], '--', label='E_mec')
ax[5].set_ylabel('[J]'); ax[5].legend(); ax[5].set_xlabel('t [s]')
for a in ax: a.grid(True, alpha=.3)
fig.suptitle('Polipasto · variables desde datos_mecanismo.csv'); fig.tight_layout()
fig.savefig('graficas_desde_csv.png', dpi=150)

## 4. Escenarios superpuestos
Corre antes `correr_escenarios.m` en MATLAB.

In [ ]:
archivos = sorted(Path('escenarios').glob('datos_*.csv'))
fig, ax = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
for f in archivos:
    d = pd.read_csv(f); nombre = f.stem.replace('datos_', '')
    ax[0].plot(d['t [s]'], d['x [m]'], label=nombre)
    ax[1].plot(d['t [s]'], d['F_op [N]'], label=nombre)
    ax[2].plot(d['t [s]'], d['P [W]'], label=nombre)
ax[1].axhline(300, ls='--', c='r', label='F_max = 300 N')
ax[0].set_ylabel('x [m]'); ax[1].set_ylabel('F_op [N]'); ax[2].set_ylabel('P [W]'); ax[2].set_xlabel('t [s]')
for a in ax: a.grid(True, alpha=.3)
ax[1].legend(fontsize=8, ncol=2); fig.tight_layout(); fig.savefig('escenarios.png', dpi=150)

In [ ]:
resumen = []
for f in archivos:
    d = pd.read_csv(f)
    resumen.append({'escenario': f.stem.replace('datos_', ''), 'F_op max [N]': d['F_op [N]'].max(),
                    'x final [m]': d['x [m]'].iloc[-1], 'W_op [J]': d['W_op [J]'].iloc[-1],
                    'alarma': bool(d['alarma_Fop [-]'].max())})
pd.DataFrame(resumen)